# Contour Plots Testing Notebook

This notebook is for manual testing of 2D contour and heatmap plots.
It uses functions from `postprocess_functions.py` and `plot_contour_functions.py`.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from ddstartup.postprocessing.postprocess_functions import (
    load_h5_to_dataframe,
    get_input_parameters,
    scale_target,
    apply_filters,
    find_latest_h5_file
)
from ddstartup.postprocessing.plot_contour_functions import (
    plot_2d_cell_mean_heatmap,
    plot_pairwise_contours,
    plot_interactive_pairwise_contours
)
from ddstartup.utils.tools import PARAM_UNITS

print("✅ Imports successful")

✅ Imports successful


## Configuration

In [2]:
# Configuration - specify directory or files

# Option 1: Automatic - find latest file in specified directory (DEFAULT)
outputs_dir = Path('../outputs')
files_to_analyze = []

latest_h5_file = find_latest_h5_file(outputs_dir)
if latest_h5_file:
    print(f"📂 Auto-detected folder: {latest_h5_file.parent.name}")
    print(f"📄 Latest file: {latest_h5_file.name}")
    files_to_analyze = [latest_h5_file]

# Option 2: Analyze all files in a specific folder
# outputs_dir = Path('../outputs/20251008_081422_parametric_T_seeded')
# files_to_analyze = sorted(outputs_dir.glob('*.h5'))
# print(f"📂 Using folder: {outputs_dir.name}")
# print(f"📄 Found {len(files_to_analyze)} file(s): {[f.name for f in files_to_analyze]}")

# Target variable to analyze
target = 'Q_fusion'  # Change this to your desired target variable
print(f"\n🎯 Target variable: {target}")

📂 Auto-detected folder: 20251009_131829_parametric_lump
📄 Latest file: ddstartup_20251009_131829_parametric_lump.h5

🎯 Target variable: Q_fusion


## Load and Prepare Data

In [3]:
# Load data
if not files_to_analyze:
    raise ValueError("No files to analyze. Please check the configuration.")

df = load_h5_to_dataframe(files_to_analyze)
print(f"📊 Loaded dataframe with shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Get input parameters
inputs = get_input_parameters(df)
print(f"\n🔧 Input parameters ({len(inputs)}): {inputs}")

# Check if target exists
if target not in df.columns:
    print(f"❌ Target '{target}' not found in dataframe")
    print(f"Available columns: {list(df.columns)}")
    raise ValueError(f"Target '{target}' not in dataframe")

# Scale target (optional)
target_scaled = scale_target(df, target)
print(f"\n📈 Target '{target}' range: [{df[target].min():.2e}, {df[target].max():.2e}]")
print(f"📊 Target statistics:")
print(df[target].describe())

TypeError: expected str, bytes or os.PathLike object, not list

## Apply Filters (Optional)

In [ ]:
# Optional: Apply filters
filters = {
    # Example: 'Q_fusion': {'min': 0},  # Only positive Q_fusion
    # Example: 'Ti_0': {'min': 5e3, 'max': 20e3},  # Temperature range
}

if filters:
    df_filtered = apply_filters(df, filters)
    print(f"🔍 Applied filters: {filters}")
    print(f"📊 Filtered dataframe shape: {df_filtered.shape} (was {df.shape})")
    df = df_filtered
else:
    print("No filters applied")

## Test 1: Single 2D Heatmap (Interpolated)

In [ ]:
# Select two input parameters for 2D plot
if len(inputs) >= 2:
    x_param = inputs[0]
    y_param = inputs[1]
    
    print(f"Creating 2D heatmap: {x_param} vs {y_param} -> {target}")
    
    # Create output directory for test plots
    test_outputs_dir = Path('../outputs/manual_test_contour')
    test_outputs_dir.mkdir(parents=True, exist_ok=True)
    
    # Plot with interpolation
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_2d_cell_mean_heatmap(
        df, 
        x=x_param, 
        y=y_param, 
        target=target,
        outputs_dir=test_outputs_dir,
        plot_name=f'heatmap_{x_param}_vs_{y_param}',
        interpolate=True,
        ax=ax
    )
    plt.show()
    
    print(f"✅ Plot saved to {test_outputs_dir}")
else:
    print("❌ Need at least 2 input parameters for 2D heatmap")

## Test 2: Single 2D Heatmap (No Interpolation)

In [ ]:
# Same plot without interpolation (cell-mean heatmap)
if len(inputs) >= 2:
    print(f"Creating cell-mean heatmap: {x_param} vs {y_param} -> {target}")
    
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_2d_cell_mean_heatmap(
        df, 
        x=x_param, 
        y=y_param, 
        target=target,
        outputs_dir=test_outputs_dir,
        plot_name=f'heatmap_discrete_{x_param}_vs_{y_param}',
        interpolate=False,
        ax=ax
    )
    plt.show()
    
    print(f"✅ Plot saved to {test_outputs_dir}")

## Test 3: Pairwise Contours (All Combinations)

In [ ]:
# Plot all pairwise combinations of input parameters
if len(inputs) >= 2:
    print(f"Creating pairwise contours for {len(inputs)} input parameters...")
    print(f"This will create {len(inputs) * (len(inputs) - 1) // 2} plots")
    
    # Limit to avoid too many plots
    max_pairs = 6  # Adjust as needed
    
    plot_pairwise_contours(
        df,
        inputs=inputs,
        target=target,
        outputs_dir=test_outputs_dir,
        max_pairs=max_pairs,
        interpolate=True,
        plot_name=f'pairwise_contours_{target}'
    )
    
    print(f"✅ Pairwise contour plots saved to {test_outputs_dir}")
else:
    print("❌ Need at least 2 input parameters for pairwise contours")

## Test 4: Interactive Pairwise Contours (Plotly)

In [ ]:
# Create interactive plotly version
if len(inputs) >= 2:
    print(f"Creating interactive pairwise contours...")
    
    plot_interactive_pairwise_contours(
        df,
        inputs=inputs,
        target=target,
        outputs_dir=test_outputs_dir,
        plot_name=f'interactive_pairwise_{target}'
    )
    
    # The function saves an HTML file
    html_file = test_outputs_dir / f'interactive_pairwise_{target}.html'
    print(f"✅ Interactive plot saved to {html_file}")
    print(f"   Open this file in a browser to interact with the plot")
else:
    print("❌ Need at least 2 input parameters for interactive contours")

## Test 5: Custom Pair Selection

In [ ]:
# Test with specific input pairs
if len(inputs) >= 3:
    # Select specific inputs to compare
    selected_inputs = inputs[:3]  # Take first 3 inputs
    
    print(f"Creating contours for selected inputs: {selected_inputs}")
    
    plot_pairwise_contours(
        df,
        inputs=selected_inputs,
        target=target,
        outputs_dir=test_outputs_dir,
        interpolate=True,
        plot_name=f'selected_pairwise_{target}'
    )
    
    print(f"✅ Selected pairwise contours saved to {test_outputs_dir}")
else:
    print("ℹ️ Skipping custom pair selection (need at least 3 inputs)")

## Summary

### Contour Plot Functions Tested:

1. ✅ **plot_2d_cell_mean_heatmap** (interpolated)
   - Creates smooth contour plots for regular grids
   - Uses cubic interpolation

2. ✅ **plot_2d_cell_mean_heatmap** (discrete)
   - Creates cell-based heatmaps without interpolation
   - Better for sparse or irregular data

3. ✅ **plot_pairwise_contours**
   - Generates all pairwise combinations of inputs
   - Saves multiple PNG files
   - Option to limit number of pairs

4. ✅ **plot_interactive_pairwise_contours**
   - Creates interactive HTML plots using Plotly
   - Allows zooming, panning, and hovering
   - Best for detailed exploration

### Key Parameters:
- `interpolate`: True for smooth contours, False for discrete cells
- `max_pairs`: Limit number of pairwise plots to avoid overload
- `plot_name`: Custom name for saved plots

### Output Files:
All plots are saved to the specified `outputs_dir` directory.